# Design / Outfit Transfer — MASK REVIEW (run_003)

Linha **experimental**, 100% **nao-generativa**. Nao altera Run 003, FLUX,
Flow 01, quality gates nem o pipeline oficial. Nao cria modelo novo.

## Etapa atual: MASK REVIEW

```
FULL BODY -> GENERATE MASKS -> VISUAL REVIEW -> [SOMENTE SE APROVADO] -> WARP / COMPOSITE
```

O warp/composicao esta **BLOQUEADO** ate aprovacao humana explicita.

Motivo: `outside_mask_pixel_difference == 0` prova que nada fora da mascara
mudou, mas **nao** prova que a mascara corresponde a roupa. As mascaras
anteriores incluiam cabelo e meias-calcas, entao A/B nao eram avaliacao
valida da transferencia.

### O que mudou

Segmentacao refeita em `scripts/chibi/design_masks.py`, com **regioes
protegidas** e discriminador cromatico medido na arte real:

| tecido | R-B | leitura |
|---|---|---|
| capa / manga | -6.7 a -12.4 | **frio** |
| meia-calca | +6.6 a +15.7 | **quente** |
| cabelo | -0.5 a -2.5 | **neutro** |

`R-B` separa capa de meia-calca, o que luminancia sozinha nao faz. O cabelo
fica no meio e por isso e isolado por **conectividade**, nao por cor.

Pecas: torso · mangas · capa esquerda · capa direita · ornamentos · inferiores.
Protegidas: cabeca (rosto/olhos/boca/chifres) · cabelo · pele · meias · sapatos.


In [ ]:
#@title 1. Ambiente e repositorio { display-mode: "form" }
#@markdown Instala as dependencias (todas permissivas) e localiza o repo.
REPO_URL = "https://github.com/BloomRX/ChibiCreate"  #@param {type:"string"}
BRANCH   = "arena/01a07ece-chibicreate"  #@param {type:"string"}
ATUALIZAR_REPO = True  #@param {type:"boolean"}
#@markdown Mantenha marcado: garante que o runtime nao rode codigo antigo.

import subprocess, sys, os
from pathlib import Path

def sh(*a):
    print("$", " ".join(a))
    subprocess.run(a, check=False)

IN_COLAB = "google.colab" in sys.modules
# scikit-image >= 0.25 basta: o codigo detecta a API por versao
# (0.25.x e 0.26 divergem em remove_small_* e ThinPlateSplineTransform).
sh(sys.executable, "-m", "pip", "-q", "install",
   "numpy", "pillow", "scikit-image>=0.25", "scipy")

ROOT = None
for c in (Path.cwd(), *Path.cwd().parents):
    if (c / "scripts" / "chibi").is_dir():
        ROOT = c
        break
if ROOT is None and Path("ChibiCreate/scripts/chibi").is_dir():
    ROOT = Path("ChibiCreate").resolve()
if ROOT is None:
    sh("git", "clone", "--branch", BRANCH, REPO_URL, "ChibiCreate")
    ROOT = Path("ChibiCreate").resolve()
os.chdir(ROOT)

# SEMPRE atualizar: um clone antigo do runtime deixaria o notebook rodando
# codigo obsoleto e reproduzindo bugs ja corrigidos.
if ATUALIZAR_REPO:
    sh("git", "fetch", "--quiet", "origin", BRANCH)
    sh("git", "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")

sys.path.insert(0, str(ROOT))

# limpar modulos ja importados, senao o Python reusa a versao velha da memoria
for _m in [m for m in sys.modules if m.startswith("scripts.chibi")]:
    del sys.modules[_m]

print("\nrepo:", ROOT)
subprocess.run(["git", "log", "--oneline", "-1"])

import numpy, PIL, skimage
print("numpy", numpy.__version__, "| pillow", PIL.__version__,
      "| scikit-image", skimage.__version__)
print("licencas: BSD-3-Clause / MIT-CMU / BSD-3-Clause  (todas comerciais)")


In [ ]:
#@title 2. Carregar Run 003 e full_body { display-mode: "form" }
#@markdown `run_003/output.png` **nao esta versionado** (`experiments/**/*.png`
#@markdown e gitignored). Faca upload dele. `full_body.png` vem do repo.
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
RUN_003_PATH = ""  #@param {type:"string"}
#@markdown Deixe vazio para abrir o seletor de upload.

import sys
from pathlib import Path
import numpy as np
from PIL import Image

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.chibi import design_transfer as dt

REF = ROOT / "characters" / CHARACTER_ID / "reference"
FULL_BODY = REF / "full_body.png"
assert FULL_BODY.exists(), f"nao encontrei {FULL_BODY}"

WORK = ROOT / "experiments" / "design_transfer" / "run_003"
(WORK / "masks").mkdir(parents=True, exist_ok=True)

base_path = Path(RUN_003_PATH) if RUN_003_PATH else None
if base_path is None or not base_path.exists():
    try:
        from google.colab import files
        print("Selecione run_003/output.png:")
        up = files.upload()
        name = next(iter(up))
        base_path = WORK / "run_003_input.png"
        base_path.write_bytes(up[name])
    except ImportError:
        raise SystemExit("Fora do Colab: preencha RUN_003_PATH.")

BASE = dt.load_rgba(base_path)
SOURCE = dt.load_rgba(FULL_BODY)
print("Run 003   :", base_path, BASE.size, BASE.mode)
print("full_body :", FULL_BODY, SOURCE.size, SOURCE.mode)

if BASE.size != SOURCE.size:
    print(f"\n[nota] resolucoes diferentes; o warp mapeia a caixa do sujeito,")
    print("       entao isso e esperado e nao impede a composicao.")


In [ ]:
#@title 3. Ver as entradas (1 = Run 003, 2 = full_body) { display-mode: "form" }
import matplotlib.pyplot as plt

def flat(img, bg=(255, 255, 255)):
    c = Image.new("RGB", img.size, bg)
    c.paste(img, (0, 0), img)
    return c

fig, ax = plt.subplots(1, 2, figsize=(11, 6))
ax[0].imshow(flat(BASE));   ax[0].set_title("1. Run 003 (base preservada)")
ax[1].imshow(flat(SOURCE)); ax[1].set_title("2. full_body (fonte do design)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
#@title 4. GENERATE MASKS — protegidas + vestuario { display-mode: "form" }
#@markdown Cada mascara combina cor, posicao, conectividade, geometria, alpha
#@markdown e **exclusao explicita** das regioes protegidas.
MIN_AREA = 120  #@param {type:"integer"}

from scripts.chibi import design_masks as dm

REVIEW = dm.review_masks(SOURCE, min_area=MIN_AREA)
MASKS = REVIEW.garment
PROTECTED = REVIEW.protected
BANNED = REVIEW.banned

total = int(np.count_nonzero(REVIEW.subject))
print("REGIOES PROTEGIDAS (a composicao nunca escreve aqui)")
for k in dm.PROTECTED_REGIONS:
    n = int(np.count_nonzero(PROTECTED[k]))
    print(f"  {k:12}{n:8d}  {100*n/total:5.2f}% do sujeito")

print("\nPECAS DE VESTUARIO")
hdr = f"  {'peca':16}{'pixels':>8}{'%subj':>8}{'overlap':>9}  rigidez"
print(hdr)
for k in dm.GARMENT_REGIONS:
    m = REVIEW.metrics[k]
    print(f"  {k:16}{m['pixels']:8d}{m['pct_subject']:8.2f}"
          f"{m['protected_overlap']:9d}  {m['rigidity']}")

print("\nmask_protected_overlap_pixels == 0 em todas?",
      all(REVIEW.metrics[k]["protected_overlap"] == 0 for k in dm.GARMENT_REGIONS))


In [ ]:
#@title 5. Cada mascara separadamente { display-mode: "form" }
fig, ax = plt.subplots(2, 3, figsize=(15, 11))
for a, name in zip(ax.ravel(), dm.GARMENT_REGIONS):
    a.imshow(MASKS[name], cmap="gray")
    m = REVIEW.metrics[name]
    a.set_title(f"{name}\n{m['pixels']} px · {m['pct_subject']:.2f}%")
    a.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 5, figsize=(18, 5))
for a, name in zip(ax, dm.PROTECTED_REGIONS):
    a.imshow(PROTECTED[name], cmap="Reds")
    a.set_title(f"PROTEGIDA: {name}"); a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
#@title 6. Overlay de CADA mascara sobre full_body { display-mode: "form" }
#@markdown Olhe peca por peca: "essa mascara cobre exatamente a peca desejada?"
OVERLAY_ALPHA = 0.55  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

fig, ax = plt.subplots(2, 3, figsize=(16, 13))
for a, name in zip(ax.ravel(), dm.GARMENT_REGIONS):
    a.imshow(flat(dm.overlay_single(SOURCE, MASKS[name],
                                    dm.GARMENT_COLORS[name], OVERLAY_ALPHA)))
    a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
#@title 7. Overlay combinado + regioes protegidas { display-mode: "form" }
ov_all = dm.overlay(SOURCE, MASKS, alpha=OVERLAY_ALPHA)
ov_prot = dm.overlay(SOURCE, PROTECTED,
                     {k: (255, 0, 0) for k in PROTECTED}, 0.5)

fig, ax = plt.subplots(1, 3, figsize=(17, 7))
ax[0].imshow(flat(SOURCE));   ax[0].set_title("1. full_body original")
ax[1].imshow(flat(ov_all));   ax[1].set_title("todas as pecas")
ax[2].imshow(flat(ov_prot));  ax[2].set_title("regioes PROTEGIDAS (vermelho)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

MASK_PATHS = {}
for name, m in MASKS.items():
    p = WORK / "masks" / f"garment_{name}.png"
    Image.fromarray((m * 255).astype(np.uint8), "L").save(p)
    MASK_PATHS[name] = p
for name, m in PROTECTED.items():
    Image.fromarray((m * 255).astype(np.uint8), "L").save(
        WORK / "masks" / f"protected_{name}.png")
ov_all.save(WORK / "masks" / "overlay_all_on_fullbody.png")
ov_prot.save(WORK / "masks" / "overlay_protected.png")
print("mascaras salvas em", (WORK / "masks").relative_to(ROOT))


In [ ]:
#@title 8. Overlay de todas as mascaras sobre a Run 003 { display-mode: "form" }
#@markdown Mapeadas pela caixa do sujeito, so para conferencia visual.
sb = REVIEW.subject_bbox
db = dt.bbox_of(dt.subject_mask(BASE))
shape = np.array(BASE).shape[:2]
warped = {k: dt.warp_affine(m.astype(float), sb, db, shape, order=0) > 0.5
          for k, m in MASKS.items()}
ov_run = dm.overlay(BASE, warped, alpha=OVERLAY_ALPHA)

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
ax[0].imshow(flat(BASE));    ax[0].set_title("Run 003 (base, intocada)")
ax[1].imshow(flat(ov_run));  ax[1].set_title("4. overlay de todas sobre Run 003")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

ov_run.save(WORK / "masks" / "overlay_all_on_run003.png")
print("salvo:", (WORK / "masks" / "overlay_all_on_run003.png").relative_to(ROOT))


In [ ]:
#@title 9. Metricas, hashes e ZIP da MASK REVIEW { display-mode: "form" }
import json, platform, shutil
import skimage

metrics_doc = {
    "stage": "mask_review",
    "generative": False,
    "character": CHARACTER_ID,
    "source": {"path": str(FULL_BODY),
               "sha256": __import__("hashlib").sha256(
                   FULL_BODY.read_bytes()).hexdigest()},
    "subject_bbox": list(REVIEW.subject_bbox),
    "protected_regions": {
        k: int(np.count_nonzero(v)) for k, v in PROTECTED.items()},
    "garment": REVIEW.metrics,
    "all_protected_overlap_zero": all(
        REVIEW.metrics[k]["protected_overlap"] == 0 for k in dm.GARMENT_REGIONS),
    "libraries": {"numpy": np.__version__, "scikit-image": skimage.__version__,
                  "python": platform.python_version()},
    "licenses": {"numpy": "BSD-3-Clause", "pillow": "MIT-CMU",
                 "scikit-image": "BSD-3-Clause", "scipy": "BSD-3-Clause"},
    "approved": False,
    "human_review_required": [
        "[HUMAN REVIEW REQUIRED] Aprovar cada mascara visualmente antes de "
        "qualquer warp ou composicao."],
}
p = WORK / "mask_review.json"
p.write_text(json.dumps(metrics_doc, indent=2, ensure_ascii=False, default=str))
print(json.dumps(metrics_doc["garment"], indent=2, ensure_ascii=False, default=str))
print("\nsalvo:", p.relative_to(ROOT))

zb = WORK.parent / "mask_review_run_003"
shutil.make_archive(str(zb), "zip", WORK)
print("ZIP:", zb.with_suffix(".zip"))
try:
    from google.colab import files
    files.download(str(zb.with_suffix(".zip")))
except Exception:
    pass


In [ ]:
#@title 10. PORTAO — warp/composicao bloqueados { display-mode: "form" }
#@markdown A proxima etapa so roda apos SUA aprovacao visual das mascaras.
#@markdown Nao marque isto sem ter olhado cada overlay acima.
MASCARAS_APROVADAS = False  #@param {type:"boolean"}

objetivo_ok = all(REVIEW.metrics[k]["protected_overlap"] == 0
                  for k in dm.GARMENT_REGIONS)
print("verificacoes objetivas (overlap zero, nao-vazias):", objetivo_ok)
print("aprovacao humana:", MASCARAS_APROVADAS)

if not MASCARAS_APROVADAS:
    print("""
=========================================================
MASK REVIEW — AGUARDANDO AVALIACAO HUMANA
=========================================================
Warp e composicao NAO serao executados.

Confira em cada overlay:
  [ ] torso cobre a gola/peca central, sem pele
  [ ] mangas cobrem as mangas sino, sem cabelo
  [ ] capa esquerda / direita cobrem o manto ate a barra
  [ ] ornamentos cobrem o ouro (ombreiras, coxas, cinto)
  [ ] inferiores cobrem o calcado
  [ ] nada toca rosto, olhos, boca, cabelo, chifres, maos, pernas

Se alguma estiver errada, relate QUAL peca e o que sobra ou falta.
""")
else:
    print("\nAprovado. A etapa de warp/composicao sera habilitada "
          "em uma proxima entrega, apos seu aval.")
